# fhe-ssm on Perseus: the Colab check

The fhe-ssm recurrent language model (405M parameters, 24 blocks) written on Perseus's ops
protocol, with 64 conversations per ciphertext at ring 2^16 (`examples/fhe_ssm_from_primitives`).

**Part 1 (any runtime, CPU is enough, about 10 minutes):** the tests, the lint, then two ticks of
the full model on Perseus's numpy fake session with the published weights, every one of the 64
conversations checked against two references: the same kernels on plain numpy (the mirror) and
the trainer's closed-form forward. It passes when every reply matches both.

**Part 2 (a GPU runtime, a long build):** builds Perseus's CUDA extension here, then a smoke test
on a live session: an encryption round trip, one linear tile and two ticks of one block. Perseus
targets 48 GB GPUs; on a smaller one the smoke test may run out of memory, and its report says
where. Runtime > Change runtime type > GPU before starting if you want Part 2.

Each part writes a JSON report under `/content/`; send those back.

In [ ]:
!git clone --depth 1 --branch fhe-ssm-port https://github.com/xelananv/perseus.git /content/perseus
%cd /content/perseus
!pip install -q -e . ruff mypy huggingface_hub

## Part 1: the port on the numpy fake

In [ ]:
import json

from examples.fhe_ssm_from_primitives import colab

machine = colab.machine()
print(json.dumps(machine, indent=1))

In [ ]:
!python -m pytest -q examples/fhe_ssm_from_primitives/tests
!python examples/fhe_ssm_from_primitives/lint.py --check

In [ ]:
paths = colab.fetch_model("/content/pbd430a")  # 1.8 GB from huggingface.co/xelananv/fhe-ssm-pbd430a
paths

In [ ]:
result = colab.fake_check(paths, ticks=2)  # 64 conversations, 24 blocks; about a minute per tick
with open("/content/fake_check.json", "w", encoding="utf-8") as f:
    json.dump({"machine": machine, **result}, f, indent=1, default=str)
print("PASS" if result["verdict"]["pass"] else "FAIL", json.dumps(result["verdict"], indent=1))

## Part 2: the CUDA build and a GPU smoke test

`colab_build.py` probes the runtime, installs only what is missing (CUDA 12.6, gcc 12, NCCL,
LibArchive and Python headers, CMake, pybind11), then runs Perseus's own build scripts. The first
cell prints the plan without running it. The build takes a long time (the patched OpenFHE, then
FIDESlib32bits, then the extension); each step's log is in `/content/perseus_build_logs/`, and a
failing step prints the end of its log.

In [ ]:
!python examples/fhe_ssm_from_primitives/colab_build.py --dry-run

In [ ]:
!python examples/fhe_ssm_from_primitives/colab_build.py

In [ ]:
!bash -c "source .cache/colab_env.sh && source scripts/local_env.sh && python -m examples.fhe_ssm_from_primitives.gpu_smoke --weights /content/pbd430a/pbd430a_weights.npz --seeds /content/pbd430a/pbd430a_seeds.json --config /content/pbd430a/pbd430a_config.json --blocks 1 --out /content/gpu_smoke.json"

## What to send back

`/content/fake_check.json` (Part 1), `/content/gpu_smoke.json` (Part 2) and, if the build failed,
the log of the failing step from `/content/perseus_build_logs/`.